# 引入套件

In [4]:
import os
import pandas as pd
import numpy as np
from finlab.dataframe import FinlabDataFrame
from finlab.backtest import sim
from finlab import data
import talib
import finlab
import logging
import openpyxl
from scipy.stats import linregress
from dotenv import load_dotenv
# 載入環境變數
load_dotenv()
finlab.login()
# 使用環境變數
# finlab.login(os.getenv('FINLAB_API_KEY'))
# 設置 finlab 數據存儲路徑


已登入（使用快取憑證）。


# 下載ETF資料

In [5]:
import pandas as pd
import time
import os
import logging
import re
from io import StringIO
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 設定日誌記錄
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def get_etf_holdings_selenium(stock_id: str) -> tuple[pd.DataFrame | None, str | None]:
    """
    Step 1: 優化爬蟲，增加代號 4 位數過濾邏輯。
    """
    # 自動補足 Pocket 網址所需的 A 後綴
    formatted_id = stock_id if stock_id.endswith('A') else f"{stock_id}A"
    url = f"https://www.pocket.tw/etf/tw/{formatted_id}/fundholding"
    
    service = Service()
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    driver = None
    try:
        driver = webdriver.Chrome(service=service, options=options)
        driver.get(url)

        # Step 2: 顯式等待
        wait = WebDriverWait(driver, 35)
        try:
            wait.until(EC.presence_of_element_located((By.XPATH, "//table//tr[td or th]")))
        except:
            logging.warning(f"[{formatted_id}] 頁面表格未出現。")
            return None, None

        # Step 3: 提取日期
        try:
            date_text = driver.find_element(By.XPATH, "//*[contains(text(), '資料日期')]").text
            update_date = date_text.replace('資料日期：', '').strip()
        except:
            update_date = datetime.now().strftime('%Y/%m/%d')

        # Step 4: 解析表格
        tables = pd.read_html(StringIO(driver.page_source))
        target_df = None
        for table in tables:
            cols = [str(c) for c in table.columns]
            if any('名稱' in c for c in cols) and any('權重' in c for c in cols):
                target_df = table
                break

        if target_df is None:
            return None, update_date

        # Step 5: 自動辨識欄位並建立 DataFrame
        mapping = {}
        for i, col_name in enumerate(target_df.columns):
            c = str(col_name)
            if '代號' in c or '代碼' in c: mapping['代號'] = i
            elif '名稱' in c: mapping['名稱'] = i
            elif '權重' in c or '比例' in c: mapping['權重'] = i
            elif '持有' in c or '股數' in c: mapping['持有數'] = i

        final_df = target_df.iloc[:, list(mapping.values())].copy()
        final_df.columns = list(mapping.keys())
        
        # Step 6: 關鍵清洗 - 僅保留 4 位數阿拉伯數字代號
        # 先轉字串、去空格，再用正則表達式篩選
        final_df['代號'] = final_df['代號'].astype(str).str.strip()
        final_df = final_df[final_df['代號'].str.match(r'^\d{4}$')]
        
        # 轉換數值與單位
        final_df['權重'] = pd.to_numeric(final_df['權重'].astype(str).str.replace('%', ''), errors='coerce')
        if '持有數' in final_df.columns:
            final_df['持有數'] = pd.to_numeric(final_df['持有數'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
            final_df['持有數'] = (final_df['持有數'] / 1000).astype(int)
        
        final_df['單位'] = '張'
        return final_df, update_date

    except Exception as e:
        logging.error(f"[{formatted_id}] 爬取錯誤: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

def update_daily_etf_data():
    """
    Step 7: 執行更新並確保 CSV 檔案數據純淨。
    """
    print("\n" + "="*60)
    print("--- 開始執行 [4位數代號清洗版] 主動式 ETF 更新任務 ---")
    print("="*60)

    etf_list = [
        '00980A', '00981A', '00982A', '00984A', '00985A', 
        '00987A', '00992A', '00993A', '00994A', '00995A'
    ]
    csv_filename = "all_etf_holdings.csv"

    # 讀取現有資料並先過濾一次舊資料（保險起見）
    if os.path.exists(csv_filename):
        existing_df = pd.read_csv(csv_filename)
        existing_df['代號'] = existing_df['代號'].astype(str).str.strip()
        existing_df = existing_df[existing_df['代號'].str.match(r'^\d{4}$')]
    else:
        existing_df = pd.DataFrame()

    newly_data = []

    print(f"啟動並行任務，Worker 數量: 4")
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(get_etf_holdings_selenium, eid): eid for eid in etf_list}

        for future in as_completed(futures):
            eid = futures[future]
            try:
                df, dt = future.result()
                if df is not None and not df.empty:
                    if not existing_df.empty:
                        check = existing_df[(existing_df['etf'] == eid) & (existing_df['日期'] == dt)]
                        if not check.empty:
                            print(f"-> {eid}: 已是最新 ({dt})")
                            continue
                    
                    print(f"-> {eid}: 獲取成功 ({dt})，符合過濾條件筆數: {len(df)}")
                    df['etf'] = eid
                    df['日期'] = dt
                    newly_data.append(df)
            except Exception as e:
                logging.error(f"-> {eid} 執行緒異常: {e}")

    # Step 8: 合併與最終存檔
    if newly_data:
        combined_df = pd.concat([existing_df] + newly_data, ignore_index=True)
        cols = ['日期', 'etf', '代號', '名稱', '權重', '持有數', '單位']
        combined_df = combined_df[[c for c in cols if c in combined_df.columns]]
        
        # 存檔前最後檢查
        combined_df['代號'] = combined_df['代號'].astype(str).str.strip()
        combined_df = combined_df[combined_df['代號'].str.match(r'^\d{4}$')]
        
        combined_df.drop_duplicates(subset=['日期', 'etf', '代號'], keep='last', inplace=True)
        combined_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"\n資料已更新並存至 {csv_filename} (已確保代號皆為4位數字)")
    else:
        print("\n沒有發現新資料。")

if __name__ == "__main__":
    update_daily_etf_data()


--- 開始執行 [4位數代號清洗版] 主動式 ETF 更新任務 ---
啟動並行任務，Worker 數量: 4
-> 00980A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 50
-> 00981A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 51
-> 00982A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 58
-> 00984A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 111
-> 00985A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 50
-> 00987A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 26
-> 00992A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 52
-> 00993A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 51
-> 00994A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 32
-> 00995A: 獲取成功 (2026/04/02)，符合過濾條件筆數: 60

資料已更新並存至 all_etf_holdings.csv (已確保代號皆為4位數字)
